In [ ]:
# base_financeiro

import pandas as pd
import numpy as np

caminho_completo = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_financeiro.csv'

df = pd.read_csv(caminho_completo) 

colunas_days = [col for col in df.columns if 'DAYS' in col]

df[colunas_days] = df[colunas_days].mask(df[colunas_days] > 0, np.nan)

print("--- Resultado após a limpeza ---")
print(df[['SK_ID_CURR'] + colunas_days].head())

caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_financeiro_limpa.csv'
df.to_csv(caminho_salvar, index=False)

print("\nPlanilha limpa salva com sucesso como 'base_financeiro_limpa.csv'!")

In [ ]:
# base_infos_pessoais

import pandas as pd
import numpy as np

caminho_completo = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_infos_pessoais.csv'
df_novo = pd.read_csv(caminho_completo)

df_novo.loc[df_novo['DAYS_BIRTH'] >= 0, 'DAYS_BIRTH'] = np.nan

mask_single = df_novo['NAME_FAMILY_STATUS'] == 'Single / Not Married'
df_novo.loc[mask_single, 'CNT_CHILDREN'] = 0
df_novo.loc[mask_single, 'CNT_FAM_MEMBERS'] = 1 + df_novo.loc[mask_single, 'CNT_CHILDREN']

mask_married = df_novo['NAME_FAMILY_STATUS'].isin(['Married', 'Civil Marriage'])
df_novo.loc[mask_married, 'CNT_FAM_MEMBERS'] = 2 + df_novo.loc[mask_married, 'CNT_CHILDREN']

df_novo.loc[df_novo['DAYS_BIRTH'] >= -6574, 'DAYS_BIRTH'] = np.nan

mask_higher = (df_novo['NAME_EDUCATION_TYPE'] == 'Higher education') & (df_novo['DAYS_BIRTH'] > -7665)
df_novo.loc[mask_higher, 'DAYS_BIRTH'] = np.nan

colunas_verificacao = ['DAYS_BIRTH', 'NAME_FAMILY_STATUS', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'NAME_EDUCATION_TYPE']
print("--- Amostra após a limpeza lógica ---")
print(df_novo[colunas_verificacao].head(10))

caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_infos_pessoais_limpa.csv'

df_novo.to_csv(caminho_salvar, index=False)

print("\nSucesso! Planilha limpa salva como 'base_infos_pessoais_limpa.csv'.")

In [ ]:
# base_regional
import pandas as pd
import numpy as np

# 1. CARREGANDO A PLANILHA REGIONAL
caminho_regional = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_regional.csv'
df_regional = pd.read_csv(caminho_regional)

# --------------------------------------------------------------------------------
# ETAPA 1: Colunas que devem estar entre 0 e 1 (Normalizadas)
# --------------------------------------------------------------------------------
colunas_0_1 = [
    'REGION_POPULATION_RELATIVE', 
    'APARTMENTS_AVG', 
    'YEARS_BEGINEXPLUATATION_AVG', 
    'ELEVATORS_AVG'
]

for col in colunas_0_1:
    # Se o valor for menor que 0 OU maior que 1, transformamos em NaN
    df_regional[col] = df_regional[col].mask((df_regional[col] < 0) | (df_regional[col] > 1), np.nan)

# --------------------------------------------------------------------------------
# ETAPA 2: Rating da Região (Deve ser entre 1 e 3)
# --------------------------------------------------------------------------------
col_rating = 'REGION_RATING_CLIENT_W_CITY'
# Se o rating for menor que 1 OU maior que 3, transformamos em NaN
df_regional[col_rating] = df_regional[col_rating].mask((df_regional[col_rating] < 1) | (df_regional[col_rating] > 3), np.nan)

# --------------------------------------------------------------------------------
# ETAPA 3: SALVANDO O RESULTADO
# --------------------------------------------------------------------------------
caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_regional_limpa.csv'
df_regional.to_csv(caminho_salvar, index=False)

# --- Verificação final no terminal ---
print("=== Relatório de Limpeza (Base Regional) ===")
print(f"Intervalo esperado 0-1:\n{df_regional[colunas_0_1].agg(['min', 'max'])}")
print(f"\nIntervalo esperado 1-3 em Rating:\nMin: {df_regional[col_rating].min()} | Max: {df_regional[col_rating].max()}")
print(f"\nArquivo salvo em: {caminho_salvar}")

In [ ]:
# base_scores
import pandas as pd
import numpy as np

# 1. CARREGANDO A BASE DE SCORES
path_scores = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_scores.csv'
df_scores = pd.read_csv(path_scores)

# --------------------------------------------------------------------------------
# ETAPA 1: Scores Externos (Devem estar entre 0 e 1)
# --------------------------------------------------------------------------------
cols_ext = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

for col in cols_ext:
    # Anula se for menor que 0 OU maior que 1
    df_scores[col] = df_scores[col].mask((df_scores[col] < 0) | (df_scores[col] > 1), np.nan)

# --------------------------------------------------------------------------------
# ETAPA 2: Mudança de Telefone (Deve ser < 0)
# --------------------------------------------------------------------------------
# Se for maior ou igual a 0, transformamos em NaN
df_scores['DAYS_LAST_PHONE_CHANGE'] = df_scores['DAYS_LAST_PHONE_CHANGE'].mask(df_scores['DAYS_LAST_PHONE_CHANGE'] >= 0, np.nan)

# --------------------------------------------------------------------------------
# ETAPA 3: SALVANDO O RESULTADO
# --------------------------------------------------------------------------------
caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_scores_limpa.csv'
df_scores.to_csv(caminho_salvar, index=False)

print("Base de scores processada e salva com sucesso!")
print(df_scores[cols_ext + ['DAYS_LAST_PHONE_CHANGE']].head())

In [ ]:
# fazendo o merge
import pandas as pd

# 1. Definindo os caminhos das bases limpas
paths = [
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/base_bens_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/base_financeiro_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/base_infos_pessoais_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/base_regional_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/base_scores_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/base_target_limpa.csv'
]

# 2. Carregando a primeira base para iniciar o "esqueleto" do dataframe final
df_final = pd.read_csv(paths[0])

# 3. Loop para ler e juntar as outras 5 bases
for path in paths[1:]:
    df_proximo = pd.read_csv(path)
    
    # Fazemos o merge usando o SK_ID_CURR. 
    # O how='left' garante que manteremos todos os IDs da base principal.
    df_final = pd.merge(df_final, df_proximo, on='SK_ID_CURR', how='left')

# 4. Verificação final
print(f"Junção concluída!")
print(f"Quantidade total de colunas: {df_final.shape[1]}")
print(f"Quantidade total de linhas: {df_final.shape[0]}")

# 5. Salvar o Dataset mestre unificado
caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/dataset_completo_unificado.csv'
df_final.to_csv(caminho_salvar, index=False)

print(f"\nArquivo mestre salvo em: {caminho_salvar}")